# Level 5 — Package the trained model (Colab Pro L4)

**Goal:** turn the finished adapter into one file that runs outside Colab, then prove that file is still as good.

## What this notebook does

1. **Build llama.cpp** (pinned release) for this GPU.
2. **Merge** the adapter into the base model → convert to **GGUF** → shrink to **Q4_K_M** (~4.7 GB).
3. **Push** the file to a private HF model repo. A re-run downloads it and skips step 2.
4. **Start the server** (OpenAI-compatible, the same kind the Chat API will call) and try 3 messages.
5. **Same 257-question exam again**, because the packaged file is not the file the exam scored.

## GPU and time

Pick **L4**. About 45–60 minutes, ~5 units (estimate). Use **Run all**. Disconnect when Cell 9 is done.
Send Cell 9's output to Claude. If a cell stops with an error, send that error.

In [ ]:
# Cell 2 — setup: clone repo + install dependencies (defensive)
REPO_URL  = "https://github.com/ankit8453/chat-AI-model.git"   # private repo -> asks for a GitHub token once
REPO_PATH = "/content/chat-AI-model"

import os, subprocess, sys, getpass

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", url, REPO_PATH], capture_output=True, text=True)

if os.path.isdir(REPO_PATH) and not os.path.isfile(os.path.join(REPO_PATH, "README.md")):
    print(f"Removing partial clone at {REPO_PATH} ...")
    subprocess.run(["rm", "-rf", REPO_PATH], check=True)
if not os.path.isdir(REPO_PATH):
    r = _clone(REPO_URL)
    if r.returncode != 0:
        print("Public clone failed (private repo?). Paste a GitHub token with read access:")
        tok = getpass.getpass("GitHub token: ")
        r = _clone(REPO_URL.replace("https://", f"https://{tok}@"))
        if r.returncode != 0:
            raise SystemExit("git clone failed:\n" + r.stderr[-1500:])
        subprocess.run(["git", "-C", REPO_PATH, "remote", "set-url", "origin", REPO_URL], check=False)
else:
    # Already cloned in this runtime -> pull the latest code (fixes pushed since the last run).
    r = subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only"], capture_output=True, text=True)
    if r.returncode != 0:
        tok = getpass.getpass("GitHub token (to pull updates): ")
        r = subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only",
                            REPO_URL.replace("https://", f"https://{tok}@"), "main"], capture_output=True, text=True)
    print("git pull:", (r.stdout or r.stderr).strip()[-300:])

print("code version:", subprocess.run(["git", "-C", REPO_PATH, "log", "-1", "--oneline"],
                                      capture_output=True, text=True).stdout.strip())
os.chdir(REPO_PATH)
sys.path.insert(0, os.path.join(REPO_PATH, "src"))
print("Repo root contents:", sorted(os.listdir(".")))

# transformers: keep Colab's version if it is new enough; peft / bitsandbytes / accelerate: always the latest,
# so they match whatever transformers Colab ships (4.x or 5.x — src/train/qlora.py handles both).
for _args in (["huggingface_hub>=0.24", "transformers>=4.45", "safetensors"],
              ["-U", "peft", "bitsandbytes", "accelerate"]):
    proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_args], capture_output=True, text=True)
    if proc.returncode != 0:
        print("PIP STDERR (tail):\n" + proc.stderr[-3000:])
        raise SystemExit("pip install failed — see tail above.")
import transformers, peft, bitsandbytes
print(f"transformers {transformers.__version__} · peft {peft.__version__} · bitsandbytes {bitsandbytes.__version__}")
print("setup ok")

In [ ]:
# Cell 3 — HF Hub login
from huggingface_hub import login, HfApi
login()  # Colab inline widget — paste your Write token

_whoami = HfApi().whoami()
assert _whoami["name"] == "ankit-iiitdmj", (
    f"HF Hub token belongs to {_whoami['name']!r}, expected 'ankit-iiitdmj'."
)
HF_NS = _whoami["name"]
print(f"HF Hub ok: user={HF_NS}")

In [ ]:
# Cell 4 — GPU check + names
import json, subprocess, torch
from huggingface_hub import HfApi, hf_hub_download
from train.qlora import BASE_MODEL

subprocess.run(["nvidia-smi"], check=False)
print()
EXPORT = "messages_2026-10-31"

VRAM = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1) if torch.cuda.is_available() else 0.0
if VRAM < 20:
    raise SystemExit(f"GPU has {VRAM} GB — pick the L4 runtime.")
CUDA_ARCH = "".join(map(str, torch.cuda.get_device_capability(0)))

ADAPTER_REPO = f"{HF_NS}/mbg-qwen7b-lora-full-{EXPORT}"
GGUF_REPO    = f"{HF_NS}/mbg-qwen7b-gguf-{EXPORT}"
EVAL_REPO    = f"{HF_NS}/mbg-eval-{EXPORT}"
GGUF_NAME    = "mbg-qwen7b-Q4_K_M.gguf"
LLAMA, MERGED, LIBS = "/content/llama.cpp", "/content/merged", "/content/convert_libs"
F16, Q4 = "/content/mbg-f16.gguf", f"/content/{GGUF_NAME}"

api = HfApi()
api.create_repo(GGUF_REPO, private=True, exist_ok=True)
print(f"GPU arch {CUDA_ARCH}\nadapter: {ADAPTER_REPO}\nGGUF repo: {GGUF_REPO}  private={api.repo_info(GGUF_REPO).private}")

In [ ]:
# Cell 5 — build llama.cpp (the slowest step, ~15 min, no output until it finishes)
import time
from serve.gguf import LLAMA_CPP_TAG, build_llama_cpp

t0 = time.time()
build_llama_cpp(LLAMA, CUDA_ARCH)
print(f"llama.cpp {LLAMA_CPP_TAG} built in {(time.time() - t0) / 60:.0f} min")

In [ ]:
# Cell 6 — merge + convert + quantise + push (skipped if the file is already on HF)
import os, shutil
from serve.gguf import convert_to_f16, merge_adapter, quantize

if GGUF_NAME in api.list_repo_files(GGUF_REPO):
    shutil.copy(hf_hub_download(GGUF_REPO, GGUF_NAME), Q4)
    print("downloaded the packaged model from HF")
else:
    print("merging ...");    merge_adapter(BASE_MODEL, ADAPTER_REPO, MERGED)
    print("converting ..."); convert_to_f16(LLAMA, MERGED, F16, LIBS)
    print("quantising ..."); quantize(LLAMA, F16, Q4)
    print("pushing ...")
    api.upload_file(path_or_fileobj=Q4, path_in_repo=GGUF_NAME, repo_id=GGUF_REPO,
                    commit_message="Q4_K_M from the full adapter")
    os.remove(F16)
size_gb = os.path.getsize(Q4) / 1024**3
print(f"{GGUF_NAME}: {size_gb:.2f} GB")
if not 4 < size_gb < 6:
    raise SystemExit("Unexpected file size — send this output to Claude.")

In [ ]:
# Cell 7 — start the server + 3 test messages
from mbg_eval.generate import generate_replies_http
from serve.gguf import start_server

def _load(name):
    p = hf_hub_download(f"{HF_NS}/mbg-sft-{EXPORT}", name, repo_type="dataset")
    with open(p, encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]

golden = _load("golden.jsonl")
URL = "http://127.0.0.1:8080"
server = start_server(LLAMA, Q4)
print("server is up")

sample = [next(r for r in golden if r["kind"] == "handover")] + [r for r in golden if r["kind"] == "answer"][:2]
for r, reply in zip(sample, generate_replies_http(URL, sample)):
    print(f"\n[expected {r['kind']}]\n  CUSTOMER: {r['messages'][-1]['content'][:160]!r}\n  REPLY   : {reply[:220]!r}")

In [ ]:
# Cell 8 — the same exam on the packaged model (results go to HF)
import tempfile, time
from mbg_eval.metrics import by_group, score, summarize

NAME = "full_gguf_q4"
t0 = time.time()
preds = generate_replies_http(URL, golden)
seconds = time.time() - t0
scores = [score(r, p) for r, p in zip(golden, preds)]
summary = summarize(scores)
summary["by_language"] = by_group(golden, scores, "lang")
summary["by_kind"] = by_group(golden, scores, "kind")
summary["seconds_per_reply_4_at_a_time"] = round(seconds / len(golden), 2)

tmp = tempfile.mkdtemp()
with open(f"{tmp}/{NAME}_preds.jsonl", "w", encoding="utf-8") as fh:
    for r, p, s in zip(golden, preds, scores):
        fh.write(json.dumps({"item_id": r["item_id"], "pred": p, **{k: s[k] for k in
                 ("passed", "problems", "expected_handover", "pred_handover")}}, ensure_ascii=False) + "\n")
with open(f"{tmp}/{NAME}_summary.json", "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, ensure_ascii=False)
api.upload_folder(folder_path=tmp, repo_id=EVAL_REPO, repo_type="dataset", commit_message=f"exam: {NAME}")
server.terminate()
print(f"exam done in {seconds / 60:.1f} min, saved to {EVAL_REPO}")

In [ ]:
# Cell 9 — comparison + verdict (reads the saved results, so it also works in a new session)
from mbg_eval.metrics import compare_table

def _summary(name):
    with open(hf_hub_download(EVAL_REPO, f"{name}_summary.json", repo_type="dataset", force_download=True),
              encoding="utf-8") as fh:
        return json.load(fh)

full, q4 = _summary("full_qlora"), _summary("full_gguf_q4")
print(compare_table({"full": full, "packaged": q4}))
print("\npass rate by language:")
for lang in sorted(q4["by_language"]):
    print(f"  {lang:<9} full {full['by_language'][lang]['pass_rate']:.2f} -> packaged {q4['by_language'][lang]['pass_rate']:.2f}")
print("\nproblems:", q4["problems"])
print(f"speed on this GPU: {q4['seconds_per_reply_4_at_a_time']} s per reply, 4 at a time (a VM without a GPU will be slower)")

ok = (q4["pass_rate"] >= full["pass_rate"] - 0.03 and q4["decision_accuracy"] >= full["decision_accuracy"] - 0.03
      and q4["handover_format_ok"] >= 0.90 and q4["grounded_rate"] >= 0.95)
print("\nVERDICT:", "READY — the packaged model is as good as the trained one" if ok else "CHECK — send this output to Claude")